In [1]:
import sys

sys.path.append("../")

In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/lichess_db_eval/lichess_db_eval.training_az73/part_000000000.parquet"
)

In [2]:
df.head()

,encoded_board,encoded_policy_target,encoded_value_target
0,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b'P\xd4'
1,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b'\x00\x00'
2,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b'\xd0c'
3,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b'\x00N'
4,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b'\xd0c'


In [2]:
import polars as pl

lazy_df = pl.scan_ndjson("../data/lichess_db_eval/lichess_db_eval.jsonl")

In [3]:
# weight_i = softmax(-δcp_i / τ)   where δcp_i = cp_best − cp_i,  τ ≈ 50–100


def cp_weighted_moves(cp_values: list[int], temperature=75):
    best_cp = cp_values[0]
    delta_cps = [best_cp - cp for cp in cp_values]
    weights = [pow(2.71828, -delta_cp / temperature) for delta_cp in delta_cps]
    total_weight = sum(weights)
    normalized_weights = [weight / total_weight for weight in weights]
    return normalized_weights


def mate_to_cp(mate_expr, cp_expr):
    """Use mate score as large cp sentinel, fall back to cp if no mate."""
    return (
        pl.when(mate_expr.is_not_null())
        .then(
            pl.when(mate_expr > 0)
            .then(30000 - mate_expr * 10)  # mate=1 → 29990, mate=3 → 29970
            .otherwise(-30000 - mate_expr * 10)  # mate=-1 → -29990, mate=-3 → -29970
        )
        .otherwise(cp_expr)
    )


def effective_cp_element():
    """For use inside list.agg (pl.element() context)."""
    return mate_to_cp(
        pl.element().struct.field("mate"),
        pl.element().struct.field("cp"),
    )


def effective_cp_single(pvs_expr):
    """For use on a single struct (e.g. after list.get(0))."""
    return mate_to_cp(
        pvs_expr.struct.field("mate"),
        pvs_expr.struct.field("cp"),
    )

In [4]:
cp_weighted_moves([-69, -163, -229, -231, -237])

[0.6150902941516558,
 0.17564002473355017,
 0.07285252399070863,
 0.07093546568655931,
 0.06548169143752612]

In [7]:
filtered_lazy_df = lazy_df.select(
    pl.col("fen"),
    pl.col("evals")
    .list.get(0)
    .struct.field("pvs")
    .list.agg(pl.element().struct.field("line").str.split(" ").list.get(0))
    .list.join(",")
    .alias("moves"),
    # cp based move weights (mate-aware)
    pl.when(pl.col("fen").str.split(" ").list.get(1) == "b")
    .then(
        pl.col("evals")
        .list.get(0)
        .struct.field("pvs")
        .list.agg(effective_cp_element() * -1)
    )
    .otherwise(
        pl.col("evals").list.get(0).struct.field("pvs").list.agg(effective_cp_element())
    )
    .map_elements(cp_weighted_moves, return_dtype=pl.List(pl.Float64))
    .alias("move_weights"),
    # Evaluation (mate-aware)
    (
        effective_cp_single(pl.col("evals").list.get(0).struct.field("pvs").list.get(0))
        * pl.when(pl.col("fen").str.split(" ").list.get(1) == "b").then(-1).otherwise(1)
    )
    .clip(-1000, 1000)
    .alias("evaluation"),
).slice(0, 15_000_000)

In [8]:
filtered_lazy_df.slice(0, 100).collect().row(0, named=True)

{'fen': '7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - -',
 'moves': 'f7g7,h8d8,h8a8,h8f8,h8b8',
 'move_weights': [0.6150902941516558,
  0.17564002473355017,
  0.07285252399070863,
  0.07093546568655931,
  0.06548169143752612],
 'evaluation': -69}